In [4]:
using Plots

In [45]:
# ================ #
#  SET PARAMETERS  #
# ================ #

alpha = 0.40;   # capital share
delta = 0.08;   # depreciation rate
beta = 0.98;    # discount factor
rho = 0.5;      # SS replacement rate

# iteration parameters
maxiter = 2000; # max number of iterations
tol = 1e-3;     # error tolerance level
adjK = 0.2;     # adjustment factor of capital in updating guess

# grid construction
Nj = 61;        # max age: 61 years max (age 20-80)
Njw = 45;       # working years (retire at Njw+1: enter at 21 and retire at 65)
Na = 201;       # asset state
Na2 = 8001;     # asset choice
Ne = 2;         # labor productivity (low/high)

minA = 0.0;       # min asset grid
maxA = 25.0;      # max asset grid
curvA = 1.2;    # asset grid density (=1 eequal size)

# labor productivity grid (mean=1)
gride = zeros(Ne);
dif = 0.3;
gride[1] = 1.0 - dif;
gride[2] = 1.0 + dif;

# labor productivity transition matrix
Pe = zeros(Ne,Ne);
Pe[1,1] = 0.8;
Pe[1,2] = 1.0 - Pe[1,1];

Pe[2,2] = copy(Pe[1,1]);
Pe[2,1] = 1.0 - Pe[2,2];


# age distribution (assume no death and no pop growth. sum to 1)
meaJ = 1/Nj .* ones(Nj);


# aggregate labor supply
L = sum(meaJ[1:Njw]);

In [46]:
# create constructer that contains parameters
struct Model{TI<:Integer, TF<:AbstractFloat}
    
    alpha::TF   # capital share
    delta::TF   # depreciation rate
    beta::TF    # discount factor
    rho::TF      # SS replacement rate

    # iteration parameters
    maxiter::TI # max number of iterations
    tol::TF     # error tolerance level
    adjK::TF     # adjustment factor of capital in updating guess

    # grid construction
    Nj::TI        # max age: 61 years max (age 20-80)
    Njw::TI       # working years (retire at Njw+1: enter at 21 and retire at 65)
    Na::TI       # asset state
    Na2::TI     # asset choice
    Ne::TI         # labor productivity (low/high)

    # minA::TF       # min asset grid
    maxA::TF      # max asset grid
    curvA::TF    # asset grid density (=1 eequal size)

    meaJ::Array{TF,1}
    L::TF
    gride::Array{TF,1}
    Pe::Array{TF,2}

    b::TF                 # borrowing limit

end


In [47]:
m = Model(
    alpha, delta, beta, rho,
    maxiter, tol, adjK,
    Nj, Njw, Na, Na2, Ne,
    # minA, 
    maxA, curvA,
    meaJ, L, gride, Pe,
    0.0
);

In [30]:
function generate_capital_grid(m, r=nothing, wage=nothing)

    if (r === nothing) && (wage === nothing) # if these are not specified, use no-borrowing condition
        phi = 0.0;
    else
        # borrowing limit
        if r <= 0.0
            phi = m.b;
        else
            phi = min(m.b,wage*m.s[1]/r);
        end
    end

    # -phi is borrowing limit, b is adhoc
    # the second term is natural limit

    # capital grid (need define in each iteration since it depends on r/phi)
    minA = -phi;                                  # borrowing constraint

    # asset state grid
    grida = zeros(Na);
    grida[1] = minA;
    for ac in 2:Na
        grida[ac] = grida[1] + (maxA-minA)*((ac-1)/(Na-1))^curvA;
    end

    # asset choice grid
    grida2 = zeros(Na2);
    grida2[1] = minA;
    for ac in 2:Na2
        grida2[ac] = grida2[1] + (maxA-minA)*((ac-1)/(Na2-1))^curvA;
    end

    return grida, grida2
end
grids = generate_capital_grid(m);

In [37]:
function translate_capital_grid(m, grids)

    grida, grida2 = grids

    # =================================================== #
    #  SPLIT GRID in grida2 TO NEARBY TWO GRIDS IN grida  #
    # =================================================== #
    
    # calculate node and weight for interpolation  
    ac1vec=zeros(Int64, m.Na2);
    ac2vec=zeros(Int64, m.Na2);

    pra1vec=zeros(m.Na2);
    pra2vec=zeros(m.Na2);

    for ac in 1:m.Na2

        xx = grida2[ac];

        if xx >= grida[m.Na]

            ac1vec[ac] = m.Na;
            ac2vec[ac] = m.Na;

            pra1vec[ac] = 1.0;
            pra2vec[ac] = 0.0;

        else

            ind = 1;
            while xx > grida[ind+1]
                ind += 1
                if ind+1 >= m.Na
                    break
                end
            end

            ac1vec[ac] = ind

            if ind < m.Na

                ac2vec[ac] = ind+1;
                dA=(xx-grida[ind])/(grida[ind+1]-grida[ind]);
                pra1vec[ac] = 1.0-dA;
                pra2vec[ac] = dA;

            else

                ac2vec[ac] = ind;
                pra1vec[ac] = 1.0;
                pra2vec[ac] = 0.0;

            end
        end
    end

    return ac1vec, ac2vec, pra1vec, pra2vec
end
capital_grid_translations = translate_capital_grid(m, grids);

In [50]:
function solve_value_function(m, grids, capital_grid_translations)

    # asset grids
    grida, grida2 = grids;

    # mapping between asset variables
    ac1vec, ac2vec, pra1vec, pra2vec = capital_grid_translations;

    # value function/solution (initialization)
    vfun = zeros(m.Nj, m.Ne, m.Na);
    afunG = zeros(Int64, m.Nj, m.Ne, m.Na);
    afun = zeros(m.Nj, m.Ne, m.Na);

    # income grid (initialization)
    yvec = zeros(m.Nj,m.Ne);

    # distribution (initialization)
    mea = zeros(m.Nj,m.Ne,m.Na)

    # equilibrium tax rate
    tau = rho*sum(m.meaJ[m.Njw+1:m.Nj])/sum(m.meaJ[1:m.Njw]);

    # initial guess of K
    K = 7.0;
    r = 0.0;

    for iter = 1:m.maxiter

        # compute r/w/SS
        r = m.alpha*(K/L)^(m.alpha-1) - m.delta;
        w = (1-m.alpha)*(K/L)^m.alpha;
        ss = m.rho * w
        
        # net income by age
        for jc in 1:m.Njw
            for ec in 1:m.Ne
                yvec[jc,ec] = (1-tau) * w * m.gride[ec]
            end
        end
        yvec[m.Njw+1:m.Nj,:] .= ss;


        # household problem: from last age Nj to 1 (backwards)
        
        # (1) age Nj (last age)
        jc = m.Nj;
        for ec in 1:m.Ne
            for ac in 1:m.Na
                
                c = yvec[jc,ec] + (1+r) * grida[ac];
                vfun[jc,ec,ac] = log(c);
                afunG[jc,ec,ac] = 1; # no saving
                afun[jc,ec,ac] = grida2[1];

            end
        end

        # (2) age Nj-1:1
        for jc in m.Nj-1:-1:1
            for ec in 1:m.Ne

                y = yvec[jc,ec];

                for ac in 1:m.Na

                    vtemp = -1000000 .* ones(m.Na2); # initialization (store values)
                    accmax = m.Na2;

                    for acc in 1:m.Na2

                        c = y + (1+r) * grida[ac] - grida2[acc];

                        if c <= 0.0
                            accmax = acc-1;
                            break
                        end

                        acc1 = ac1vec[acc];
                        acc2 = ac2vec[acc];

                        vpr = 0.0;
                        for ecc in 1:m.Ne
                            vpr += m.Pe[ec,ecc] * (
                                pra1vec[acc] * vfun[jc+1,ecc,acc1] 
                                + pra2vec[acc] * vfun[jc+1,ecc,acc2])
                                ;
                        end

                        vtemp[acc] = log(c) + m.beta * vpr;

                    end

                    val,index = findmax(vtemp[1:accmax]);
                    vfun[jc,ec,ac] = val;
                    afunG[jc,ec,ac] = index; # grid from grida2
                    afun[jc,ec,ac] = grida2[index];

                end
            end
        end

        # compute distribution mea
        mea = zeros(m.Nj,m.Ne,m.Na);    # initialization
        mea[1,:,1] .= m.meaJ[1]/m.Ne; # zero asset at age 1

        for jc in 1:m.Nj-1
            for ec in 1:m.Ne
                for ac in 1:m.Na

                    mea0 = mea[jc,ec,ac];
                    acc = afunG[jc,ec,ac];
                    
                    acc1 = ac1vec[acc];
                    acc2 = ac2vec[acc];

                    pra1 = pra1vec[acc];
                    pra2 = pra2vec[acc];

                    for ecc in 1:m.Ne
                        
                        mea[jc+1,ecc,acc1] += m.Pe[ec,ecc]*pra1*mea0;
                        mea[jc+1,ecc,acc2] += m.Pe[ec,ecc]*pra2*mea0;

                    end
                end
            end
        end

        errm = abs(sum(mea)-1);
        if errm > 1e-4
            println("error in computation of distribution: $errm")
            break
        end

        mea_maxA = sum(mea[:,:,m.Na]);
        if mea_maxA > 1e-3
            println("measure at max asset grid is large: $mea_maxA")
        end

        # compute error in K=A
        A = sum(afun.*mea);
        errK = abs(K-A);

        if errK < m.tol
            println("K coverged: $iter")
            break
        end

        if iter > m.maxiter
            println("WARN: iter>maxiter: $iter, $errK")
        end

        # update Kitao
        K += m.adjK*(A-K);

        println("$iter errK = $errK")

    end

    return vfun, afun, mea, r, w, ss, K

end


solve_value_function (generic function with 1 method)

In [51]:
res = solve_value_function(m, grids, capital_grid_translations);

1 errK = 2.9561332195148493
2 errK = 0.9493548131707792
3 errK = 0.2203752176811795
4 errK = 0.045122494864247464
5 errK = 0.00835506937669983
6 errK = 0.0019005269091634247
K coverged: 7


LoadError: UndefVarError: `w` not defined in `Main`
Suggestion: check for spelling errors or missing imports.